In [2]:
from pyspark import SparkContext
print(SparkContext._active_spark_context)  

None


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("iceberg-query-session")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.machine",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.machine.type",
        "hive"
    )
    .config(
        "spark.sql.catalog.machine.uri",
        "thrift://hive-metastore:9083"
    )
    .config(
        "spark.sql.catalog.machine.warehouse",
        "/data/warehouse"
    )
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jmattam/.ivy2/cache
The jars for the packages stored in: /home/jmattam/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2c8dd631-1ab0-4cb5-8fbc-adaed2942478;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
:: resolution report :: resolve 102ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.

In [4]:
spark.sql("SELECT * FROM machine.silver.agg_raw_log LIMIT 5").show()

+-------------+--------------------+-----+----------------+--------+----------------------+
|    ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-------------+--------------------+-----+----------------+--------+----------------------+
|CHS86CECE4B6D|2021-05-25 04:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|    119097.15000000001|
|CHS86CECE4B6D|2021-05-30 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|              2812.403|
|CHS66919BA506|2021-05-29 22:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|    123625.40000000001|
|CHS66919BA506|2021-05-30 02:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|    124255.45000000001|
|CHS86CECE4B6D|2021-05-29 15:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|             137831.45|
+-------------+--------------------+-----+----------------+--------+----------------------+



In [5]:
spark.table("machine.silver.agg_raw_log").limit(5).show()

+-------------+--------------------+-----+----------------+--------+----------------------+
|    ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-------------+--------------------+-----+----------------+--------+----------------------+
|CHS86CECE4B6D|2021-05-25 04:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|    119097.15000000001|
|CHS86CECE4B6D|2021-05-30 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|              2812.403|
|CHS66919BA506|2021-05-29 22:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|    123625.40000000001|
|CHS66919BA506|2021-05-30 02:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|    124255.45000000001|
|CHS86CECE4B6D|2021-05-29 15:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|             137831.45|
+-------------+--------------------+-----+----------------+--------+----------------------+



#### See full history

In [8]:
spark.sql("SELECT * FROM machine.silver.agg_raw_log.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                           |summary                                                                                                                                                                                                                                

#### History - Lineage of all commits

In [9]:
spark.sql("SELECT * FROM machine.silver.agg_raw_log.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-06 21:10:25.093|1986763261127152283|NULL               |true               |
|2026-09-07 19:38:16.83 |1995760817468641714|1986763261127152283|true               |
+-----------------------+-------------------+-------------------+-------------------+



#### Time travel by Snapshot ID & Timestamp

In [17]:
spark.sql("delete from  machine.silver.agg_raw_log where CHassisID = 'CHS86CECE4B6D' and MEASURE_CODE_ACC = 'TOTAL_FUEL_USED' and CREATIONDATETIME <='2021-05-25 00:00:00'")

DataFrame[]

In [18]:
spark.sql("SELECT * FROM machine.silver.agg_raw_log.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-06 21:10:25.093|1986763261127152283|NULL               |true               |
|2026-09-07 19:38:16.83 |1995760817468641714|1986763261127152283|true               |
|2026-09-07 19:44:50.874|672526282769054088 |1995760817468641714|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [20]:
## Current Value All values before 25th May deleted
spark.sql("""
    SELECT CHASSISID, DATE_TRUNC('DAY',CREATIONDATETIME) as CREATED_DATE,SUM(MEASURE_CODE_ACC_VALUE) AS TOTAL_FUEL_USED 
    FROM machine.silver.agg_raw_log where CHassisID = 'CHS86CECE4B6D' and MEASURE_CODE_ACC = 'TOTAL_FUEL_USED'
    group by 1,2 order by 2 asc
""").show()

+-------------+-------------------+-----------------+
|    CHASSISID|       CREATED_DATE|  TOTAL_FUEL_USED|
+-------------+-------------------+-----------------+
|CHS86CECE4B6D|2021-05-25 00:00:00|         66995.68|
|CHS86CECE4B6D|2021-05-26 00:00:00|80036.03999999998|
|CHS86CECE4B6D|2021-05-27 00:00:00|        123511.06|
|CHS86CECE4B6D|2021-05-28 00:00:00|         60652.49|
|CHS86CECE4B6D|2021-05-29 00:00:00|69185.62999999999|
|CHS86CECE4B6D|2021-05-30 00:00:00|61643.75000000001|
|CHS86CECE4B6D|2021-05-31 00:00:00|88077.55999999998|
+-------------+-------------------+-----------------+



In [21]:
## By snapshot ID  -- All records present
spark.sql("""
    SELECT CHASSISID, DATE_TRUNC('DAY',CREATIONDATETIME) as CREATED_DATE,SUM(MEASURE_CODE_ACC_VALUE) AS TOTAL_FUEL_USED 
    FROM machine.silver.agg_raw_log  VERSION AS OF 1986763261127152283 where CHassisID = 'CHS86CECE4B6D' and MEASURE_CODE_ACC = 'TOTAL_FUEL_USED'
    group by 1,2 order by 2 asc
""").show()

+-------------+-------------------+------------------+
|    CHASSISID|       CREATED_DATE|   TOTAL_FUEL_USED|
+-------------+-------------------+------------------+
|CHS86CECE4B6D|2021-05-24 00:00:00| 53976.13400000001|
|CHS86CECE4B6D|2021-05-25 00:00:00| 66995.63999999998|
|CHS86CECE4B6D|2021-05-26 00:00:00|         80036.056|
|CHS86CECE4B6D|2021-05-27 00:00:00|123511.03300000002|
|CHS86CECE4B6D|2021-05-28 00:00:00|         60652.473|
|CHS86CECE4B6D|2021-05-29 00:00:00| 69185.60900000001|
|CHS86CECE4B6D|2021-05-30 00:00:00|          61643.78|
|CHS86CECE4B6D|2021-05-31 00:00:00|          88077.54|
+-------------+-------------------+------------------+



In [23]:
## By timestamp  -- All records present
spark.sql("""
    SELECT CHASSISID, DATE_TRUNC('DAY',CREATIONDATETIME) as CREATED_DATE,SUM(MEASURE_CODE_ACC_VALUE) AS TOTAL_FUEL_USED 
    FROM machine.silver.agg_raw_log  TIMESTAMP AS OF '2026-09-06 23:10:25.093' where CHassisID = 'CHS86CECE4B6D' and MEASURE_CODE_ACC = 'TOTAL_FUEL_USED'
    group by 1,2 order by 2 asc
""").show()


+-------------+-------------------+------------------+
|    CHASSISID|       CREATED_DATE|   TOTAL_FUEL_USED|
+-------------+-------------------+------------------+
|CHS86CECE4B6D|2021-05-24 00:00:00| 53976.13400000001|
|CHS86CECE4B6D|2021-05-25 00:00:00| 66995.63999999998|
|CHS86CECE4B6D|2021-05-26 00:00:00|         80036.056|
|CHS86CECE4B6D|2021-05-27 00:00:00|123511.03300000002|
|CHS86CECE4B6D|2021-05-28 00:00:00|         60652.473|
|CHS86CECE4B6D|2021-05-29 00:00:00| 69185.60900000001|
|CHS86CECE4B6D|2021-05-30 00:00:00|          61643.78|
|CHS86CECE4B6D|2021-05-31 00:00:00|          88077.54|
+-------------+-------------------+------------------+



#### Difference between 2 snapshots

In [30]:
spark.read \
.format("iceberg") \
.option("start-snapshot-id", "1995760817468641714") \
.option("end-snapshot-id", "672526282769054088") \
.load("machine.silver.agg_raw_log.changes").show()


+-------------+--------------------+-----+----------------+--------+----------------------+------------+---------------+-------------------+
|    ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|_change_type|_change_ordinal|_commit_snapshot_id|
+-------------+--------------------+-----+----------------+--------+----------------------+------------+---------------+-------------------+
|CHS86CECE4B6D|2021-05-25 04:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|             119097.15|      INSERT|              0| 672526282769054088|
|CHS86CECE4B6D|2021-05-30 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|                2812.4|      INSERT|              0| 672526282769054088|
|CHS66919BA506|2021-05-29 22:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|              123625.4|      INSERT|              0| 672526282769054088|
|CHS66919BA506|2021-05-30 02:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|             124255.45|      INSERT|              0| 672526282769054088|
|CHS86CECE4B6

#### Files — what physically backs the current snapshot

In [31]:
spark.sql("SELECT * FROM machine.silver.agg_raw_log.files").show(truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------+-----------+-------+------------+------------------+------------------------------------------------------------+------------------------------------------------------------------+------------------------------------------------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------+------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------

#### Query all manifest files

In [32]:
spark.sql("SELECT * FROM machine.silver.agg_raw_log.manifests").show(truncate=False)

+-------+------------------------------------------------------------------------------------------------+------+-----------------+------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|content|path                                                                                            |length|partition_spec_id|added_snapshot_id |added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|
+-------+------------------------------------------------------------------------------------------------+------+-----------------+------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|0      |file:/data/wareh